In [4]:
import os, sys

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'
os.environ['PYSPARK_PYTHON'] = sys.executable

print('Python:', sys.version)
print('JAVA_HOME:', os.environ.get('JAVA_HOME'))

Python: 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]
JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64


In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FraudToMongo") \
    .config("spark.mongodb.write.connection.uri",
            "mongodb://localhost:27017/fraud_db.ccFraud") \
    .config("spark.jars.packages",
           "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0")\
    .getOrCreate()

spark.version

:: loading settings :: url = jar:file:/home/vboxuser/spark-env/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/vboxuser/.ivy2/cache
The jars for the packages stored in: /home/vboxuser/.ivy2/jars
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d98f0ad2-df5e-45e9-af9e-8ce48791d1f6;1.0
	confs: [default]
	found org.mongodb.spark#mongo-spark-connector_2.12;10.3.0 in central
	found org.mongodb#mongodb-driver-sync;4.8.2 in central
	[4.8.2] org.mongodb#mongodb-driver-sync;[4.8.1,4.8.99)
	found org.mongodb#bson;4.8.2 in central
	found org.mongodb#mongodb-driver-core;4.8.2 in central
	found org.mongodb#bson-record-codec;4.8.2 in central
:: resolution report :: resolve 1791ms :: artifacts dl 7ms
	:: modules in use:
	org.mongodb#bson;4.8.2 from central in [default]
	org.mongodb#bson-record-codec;4.8.2 from central in [default]
	org.mongodb#mongodb-driver-core;4.8.2 from central in [default]
	org.mongodb#mongodb-driver-sync;4.8.2 from central in [default]
	org.mongodb.spark#mongo-spark-connec

'3.5.0'

In [6]:
#Read csv file
df = spark.read.csv("/home/vboxuser/Big Data Systems Design/Assignment3- Credit Card Fraud Detection/fraudTest.csv",
                    header = True,
                    inferSchema = True)
df.show(1)


+---+---------------------+----------+--------------------+-------------+----+-----+-------+------+-----------------+--------+-----+-----+-------+--------+--------+-------------------+----------+--------------------+----------+---------+----------+--------+
|ID_|trans_date_trans_time|    cc_num|            merchant|     category| amt|first|   last|gender|           street|    city|state|  zip|    lat|    long|city_pop|                job|       dob|           trans_num| unix_time|merch_lat|merch_long|is_fraud|
+---+---------------------+----------+--------------------+-------------+----+-----+-------+------+-----------------+--------+-----+-----+-------+--------+--------+-------------------+----------+--------------------+----------+---------+----------+--------+
|  0|  2020-06-21 12:14:00|2.29116E15|fraud_Kirlin and ...|personal_care|2.86| Jeff|Elliott|     M|351 Darlene Green|Columbia|   SC|29209|33.9659|-80.9355|  333497|Mechanical engineer|1968-03-19|2da90c7d74bd46a0c...|1371816865

In [7]:
df.printSchema()

root
 |-- ID_: integer (nullable = true)
 |-- trans_date_trans_time: timestamp (nullable = true)
 |-- cc_num: double (nullable = true)
 |-- merchant: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amt: double (nullable = true)
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: integer (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- city_pop: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- trans_num: string (nullable = true)
 |-- unix_time: integer (nullable = true)
 |-- merch_lat: double (nullable = true)
 |-- merch_long: double (nullable = true)
 |-- is_fraud: integer (nullable = true)



In [8]:
# Show counts for: rows and columns
print("Rows:", df.count())
print("Columns:", len(df.columns))

Rows: 555719
Columns: 23


In [9]:
print(df.columns)

['ID_', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud']


In [10]:
# import all functions
from pyspark.sql.functions import *

#Identify Columns of Interest
df_basic = df.select(
    "trans_date_trans_time",
    "cc_num",
    "merchant",
    "category",
    "amt",
    "first",
    "last",
    "gender",
    "street",
    "city",
    "state",
    "zip",
    "lat",
    "long",
    "city_pop",
    "job",
    "dob",
    "trans_num",
    "unix_time",
    "merch_lat",
    "merch_long",
    "is_fraud"
)

df_basic.show(5)

+---------------------+----------+--------------------+--------------+-----+------+--------+------+--------------------+----------+-----+-----+-------+--------+--------+--------------------+----------+--------------------+----------+---------+-----------+--------+
|trans_date_trans_time|    cc_num|            merchant|      category|  amt| first|    last|gender|              street|      city|state|  zip|    lat|    long|city_pop|                 job|       dob|           trans_num| unix_time|merch_lat| merch_long|is_fraud|
+---------------------+----------+--------------------+--------------+-----+------+--------+------+--------------------+----------+-----+-----+-------+--------+--------+--------------------+----------+--------------------+----------+---------+-----------+--------+
|  2020-06-21 12:14:00|2.29116E15|fraud_Kirlin and ...| personal_care| 2.86|  Jeff| Elliott|     M|   351 Darlene Green|  Columbia|   SC|29209|33.9659|-80.9355|  333497| Mechanical engineer|1968-03-19|2da9

In [11]:
df_clean = df_basic

In [12]:
df_clean.printSchema()

root
 |-- trans_date_trans_time: timestamp (nullable = true)
 |-- cc_num: double (nullable = true)
 |-- merchant: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amt: double (nullable = true)
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: integer (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- city_pop: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- trans_num: string (nullable = true)
 |-- unix_time: integer (nullable = true)
 |-- merch_lat: double (nullable = true)
 |-- merch_long: double (nullable = true)
 |-- is_fraud: integer (nullable = true)



In [13]:
# Create a single row data frame showing the number of null values in each column of interest
null_counts = df_clean.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_basic.columns])

#print full results with column names
null_counts.show(truncate=False)

[Stage 7:>                                                          (0 + 4) / 4]

+---------------------+------+--------+--------+---+-----+----+------+------+----+-----+---+---+----+--------+---+---+---------+---------+---------+----------+--------+
|trans_date_trans_time|cc_num|merchant|category|amt|first|last|gender|street|city|state|zip|lat|long|city_pop|job|dob|trans_num|unix_time|merch_lat|merch_long|is_fraud|
+---------------------+------+--------+--------+---+-----+----+------+------+----+-----+---+---+----+--------+---+---+---------+---------+---------+----------+--------+
|0                    |0     |0       |0       |0  |0    |0   |0     |0     |0   |0    |0  |0  |0   |0       |0  |0  |0        |0        |0        |0         |0       |
+---------------------+------+--------+--------+---+-----+----+------+------+----+-----+---+---+----+--------+---+---+---------+---------+---------+----------+--------+



In [14]:
# not all the features in the dataset are numerical. I have to conver the categorical and time feature to numerical
from pyspark.ml.feature import StringIndexer

categorical_cols = ["cc_num", "merchant", "category", "first", "last", "gender",
                    "street", "city", "state", "job"]

indexers = [StringIndexer(inputCol=c, outputCol=c+"_index", handleInvalid='keep') 
            for c in categorical_cols]

from pyspark.ml import Pipeline
pipeline = Pipeline(stages=indexers)
df_indexed = pipeline.fit(df_clean).transform(df_clean)

from pyspark.sql.functions import unix_timestamp, col

# convert date/time to number
df_indexed = df_indexed.withColumn("trans_date_trans_time_ts", unix_timestamp(col("trans_date_trans_time"), "yyyy-MM-dd HH:mm:ss"))
df_indexed = df_indexed.withColumn("dob_ts", unix_timestamp(col("dob"), "yyyy-MM-dd"))

#"trans_num"=> remove this from the set, because it is a unique identifier
numeric_features = ["trans_date_trans_time_ts", "dob_ts", "amt", "lat", "long", 
                    "city_pop", "unix_time", "merch_lat", "merch_long"] + \
                   [c+"_index" for c in categorical_cols]

from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(
    inputCols=numeric_features,
    outputCol="features"
)

#check if any column is not numerical
# df_indexed.select(numeric_features).printSchema()

dfFeatures = assembler.transform(df_indexed)
dfModel = dfFeatures.select("features", "is_fraud")
dfModel.show(5)


26/03/18 20:31:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+--------+
|            features|is_fraud|
+--------------------+--------+
|[1.59274164E9,-5....|       0|
|[1.59274164E9,6.3...|       0|
|[1.59274164E9,2.5...|       0|
|[1.5927417E9,5.54...|       0|
|[1.5927417E9,-4.5...|       0|
+--------------------+--------+
only showing top 5 rows



In [15]:
#Split data set
trainDf, testDf = dfModel.randomSplit([0.8, 0.2], seed=42)

print("Training rows:", trainDf.count())
print("Testing rows:", testDf.count())

Training rows: 444594


[Stage 44:=============================>                            (2 + 2) / 4]

Testing rows: 111125


In [16]:
#Handle imbalance
fraud_count = trainDf.filter("is_fraud = 1").count()
nonfraud_count = trainDf.filter("is_fraud = 0").count()

ratio = nonfraud_count / fraud_count


from pyspark.sql.functions import when

trainDf = trainDf.withColumn(
    "classWeight",
    when(trainDf.is_fraud == 1, ratio).otherwise(1.0)
)

In [17]:
#logistical Regression
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="is_fraud",
    weightCol="classWeight"
)

lr_model = lr.fit(trainDf)

In [18]:
#Random Forest
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="is_fraud",
    numTrees=50,
    maxBins = 1000
)

rf_model = rf.fit(trainDf)

26/03/18 20:33:45 WARN DAGScheduler: Broadcasting large task binary with size 1084.3 KiB
26/03/18 20:33:50 WARN DAGScheduler: Broadcasting large task binary with size 1780.6 KiB
26/03/18 20:33:57 WARN DAGScheduler: Broadcasting large task binary with size 2.8 MiB
                                                                                

In [20]:
#SVM Model
from pyspark.ml.classification import LinearSVC

svm = LinearSVC(
    featuresCol="features",
    labelCol="is_fraud"
)

svm_model = svm.fit(trainDf)


In [58]:
# #XGBOOST
# from xgboost.spark import SparkXGBClassifier

# xgb = SparkXGBClassifier(
#     features_col="features",
#     label_col="is_fraud",
#     num_workers=2,
#     max_depth=5,
#     eta=0.3
# )

# xgb_model = xgb.fit(trainDf)

ModuleNotFoundError: No module named 'xgboost'

In [21]:
#EVALUATE
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator_auc = BinaryClassificationEvaluator(
    labelCol="is_fraud",
    metricName="areaUnderROC"
)

evaluator_precision = MulticlassClassificationEvaluator(
    labelCol="is_fraud",
    metricName="weightedPrecision"
)

evaluator_recall = MulticlassClassificationEvaluator(
    labelCol="is_fraud",
    metricName="weightedRecall"
)


In [22]:
def evaluate_model(model, name):
    predictions = model.transform(testDf)
    
    auc = evaluator_auc.evaluate(predictions)
    precision = evaluator_precision.evaluate(predictions)
    recall = evaluator_recall.evaluate(predictions)
    
    return (name, auc, precision, recall)

results = []

results.append(evaluate_model(lr_model, "Logistic Regression"))
results.append(evaluate_model(rf_model, "Random Forest"))
results.append(evaluate_model(svm_model, "SVM"))
# results.append(evaluate_model(xgb_model, "XGBoost"))
# results.append(evaluate_model(lgbm_model, "LightGBM"))

26/03/18 20:35:20 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/03/18 20:35:26 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/03/18 20:35:31 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
                                                                                

In [24]:
results_df = spark.createDataFrame(results, ["Model", "AUC", "Precision", "Recall"])
results_df.show(truncate=False)

+-------------------+------------------+------------------+------------------+
|Model              |AUC               |Precision         |Recall            |
+-------------------+------------------+------------------+------------------+
|Logistic Regression|0.8718591603878034|0.9950503948927313|0.9249853768278965|
|Random Forest      |0.9685892833768411|0.9967082246622947|0.9970033745781777|
|SVM                |0.6208601794694185|0.9921325208567151|0.996058492688414 |
+-------------------+------------------+------------------+------------------+



## STREAM THE DATASET INTO THE MODEL: RANDOM FOREST

## Analysis using Dataframe

In [29]:
def write_to_mongo(batch_df, batch_id):
    batch_df.write \
        .format("mongodb") \
        .mode("append") \
        .save()

mongo_query = DisasterStream.writeStream \
    .foreachBatch(write_to_mongo) \
    .option("checkpointLocation", "/tmp/mongo_checkpoint") \
    .outputMode("append") \
    .start()

26/03/04 18:34:53 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [30]:
spark.streams.active

In [31]:
mongo_query.status

{'message': 'Processing new data',
 'isDataAvailable': True,
 'isTriggerActive': True}

In [32]:
# Using local filesystem to simulate distributed storage
hdfs_query = DisasterStream.writeStream \
    .format("parquet") \
    .option("path", "/home/vboxuser/Big Data Systems Design/Individual Project/disaster_archive") \
    .option("checkpointLocation", "/tmp/hdfs_checkpoint") \
    .outputMode("append") \
    .start()

26/03/04 18:35:04 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/03/04 18:35:08 ERROR FileFormatWriter: Aborting job 60c764a1-2fca-4086-ac27-b073afb46892.
org.apache.spark.SparkFileNotFoundException: [BATCH_METADATA_NOT_FOUND] Unable to find batch /home/vboxuser/Big Data Systems Design/Individual Project/disaster_archive/_spark_metadata/79.compact.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.batchMetadataFileNotFoundError(QueryExecutionErrors.scala:1848)
	at org.apache.spark.sql.execution.streaming.HDFSMetadataLog.applyFnToBatchByStream(HDFSMetadataLog.scala:194)
	at org.apache.spark.sql.execution.streaming.CompactibleFileStreamLog.applyFnInBatch(CompactibleFileStreamLog.scala:207)
	at org.apache.spark.sql.execution.streaming.CompactibleFileStreamLog.foreachInBatch(CompactibleFileStreamLog.scala:193)
	at org.apache.spark.sql.execution.streaming.CompactibleFileStreamLog.$anonfun$compact$3(Compact

In [33]:
#stop active stream
for q in spark.streams.active:
    q.stop()

In [34]:
#spark.version
mongo_query.stop()


In [35]:
hdfs_query.stop()